In [ ]:
# ==========================================
# Cell 1: 基础设置与库导入
# ==========================================
import os
import numpy as np
import pandas as pd
from scipy import signal
import wfdb
import neurokit2 as nk
import networkx as nx           # 复杂网络分析库
import hrvanalysis as hrv       # HRV及异常心拍清洗库
from tqdm import tqdm
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from pyrqa.time_series import TimeSeries
from pyrqa.settings import Settings
from pyrqa.analysis_type import Classic
from pyrqa.neighbourhood import FixedRadius
from pyrqa.metric import EuclideanMetric
from pyrqa.computation import RQAComputation, RPComputation

# ================= 基础设置 =================
# 请确保该路径正确
# os.chdir('../paf-prediction-challenge-database-1.0.0/')
sampling_frequency = 128
signal_len = 5 * 128            # 截取片段的理论长度

In [ ]:
# ==========================================
# Cell 2: 信号精细化清洗函数
# ==========================================
def clean_rr(rr_list, remove_invalid=True, low_rr=0.2, high_rr=4, interpolation_method="linear", remove_ecto=True) -> np.ndarray:
    """清理并插值 RR 间期信号，确保信号连续性满足复杂网络要求"""
    if remove_invalid:
        # 剔除超出生理范围(0.2s - 4.0s)的异常RR间期，并采用线性插值填补
        rr_list = [rr if high_rr >= rr >= low_rr else np.nan for rr in rr_list]
        rr_list = pd.Series(rr_list).interpolate(method=interpolation_method).tolist()
    if remove_ecto:
        # 使用 hrvanalysis 库识别并移除异位搏动(Ectopic beats)
        rr_list = hrv.remove_ectopic_beats(rr_list, method='custom', custom_removing_rule=0.3, verbose=False)
        # 双向插值确保序列首尾无缺失值
        rr_list = pd.Series(rr_list).interpolate(method=interpolation_method).interpolate(limit_direction='both').tolist()
    return np.array(rr_list)

def bandpass_filter(signal_data, lowcut=0.5, highcut=40, fs=128, order=4):
    """带通滤波器，用于去除基线漂移与高频噪声（肌电干扰等）"""
    nyquist = 0.5 * fs
    low = lowcut / nyquist
    high = highcut / nyquist
    b, a = signal.butter(order, [low, high], btype='band')
    return signal.filtfilt(b, a, signal_data)

In [ ]:
# ==========================================
# Cell 3: 提取用于构建复杂网络的长 RR 序列
# ==========================================
def extract_rr_intervals(data_dir, save_dir):
    """从 wfdb 数据集提取并保存 NSR 和 PreAF 的长 RR 间期 (长度=300)"""
    nsr_rri = []
    preaf_rri = []
    
    print("开始提取 NSR (正常) RR间期...")
    for i in tqdm(range(1, 51, 2)):
        record_name = os.path.join(data_dir, f'p{i:02d}')
        annotation = wfdb.rdann(record_name, 'qrs')
        rr_intervals = clean_rr(np.diff(annotation.sample) / sampling_frequency)
        
        # 复杂网络分析通常需要较长序列，此处取长度为300
        for j in range(3):
            nsr_rri.append(rr_intervals[31*j:31*(j+1)]) 
            
    print("开始提取 PreAF (房颤前) RR间期...")
    for i in tqdm(range(2, 51, 2)):
        record_name = os.path.join(data_dir, f'p{i:02d}')
        annotation = wfdb.rdann(record_name, 'qrs')
        rr_intervals = clean_rr(np.diff(annotation.sample) / sampling_frequency)
        
        for j in range(3):
            start_index = len(rr_intervals) - 300 * (j + 1)
            end_index = len(rr_intervals) - 300 * j
            if start_index < 0:
                break
            preaf_rri.append(rr_intervals[start_index:end_index])
            
    # 统一保存为 numpy 数组
    os.makedirs(save_dir, exist_ok=True)
    np.save(os.path.join(save_dir, 'NSR_RR_300.npy'), nsr_rri)
    np.save(os.path.join(save_dir, 'PreAF_RR_300.npy'), preaf_rri)
    print("提取完毕！")


In [ ]:
# ==========================================
# Cell 4: 提取网络拓扑结构与基础动力学融合特征 (核心实现)
# ==========================================
def compute_kth_moment(adj_matrix, k):
    """计算邻接矩阵密度谱的 k 阶矩，用于衡量网络的频谱复杂性"""
    # 求解邻接矩阵的所有特征值
    eigenvalues = np.linalg.eigvals(adj_matrix)
    eigenvalues_k = np.power(eigenvalues, k)
    n = adj_matrix.shape[0]
    return (np.sum(eigenvalues_k) / n).real

def get_rqa_all(file_path, threshold=0.24):
    """结合递归图和复杂网络提取 7 维综合融合特征"""
    rqa = []
    data = np.load(file_path, allow_pickle=True)
    
    for i in tqdm(range(len(data)), desc=f"处理 {os.path.basename(file_path)}"):
        time_series = TimeSeries(data[i], embedding_dimension=3, time_delay=2)
        settings = Settings(time_series, analysis_type=Classic, 
                            neighbourhood=FixedRadius(threshold), 
                            similarity_measure=EuclideanMetric, theiler_corrector=1)
        
        # 1. 提取复杂网络(RCN)全局拓扑特征
        computation_rp = RPComputation.create(settings, verbose=False)
        result_rp = computation_rp.run()
        # 将生成的递归矩阵直接视为复杂网络的无向无权邻接矩阵
        matrix = result_rp.recurrence_matrix 
        
        G = nx.from_numpy_array(matrix)
        # 计算全局拓扑指标：平均度、平均聚类系数、密度谱三阶矩
        average_degree = sum(dict(G.degree()).values()) / len(G.nodes)
        avg_clustering = nx.average_clustering(G)
        moment_3th = compute_kth_moment(matrix, 3)

        # 2. 提取基础 RQA 局部演化特征
        computation_rqa = RQAComputation.create(settings, verbose=False)
        result_rqa = computation_rqa.run()
        
        # 将 RCN特征(3维) 与 RQA特征(4维) 组合成 7 维特征向量
        rqa.append([
            average_degree, avg_clustering, moment_3th, 
            result_rqa.recurrence_rate, result_rqa.entropy_diagonal_lines, 
            result_rqa.determinism, result_rqa.laminarity
        ])
        
    return np.array(rqa)

In [ ]:
# ==========================================
# Cell 5: 模型训练、融合评估与最终预测 (对应表4-3)
# ==========================================
# 注意修改为自己电脑内的实际相对路径
# 提取特征
X_nsr = get_rqa_all('./pro_data/NSR_RR_300.npy', threshold=0.24)
X_preaf = get_rqa_all('./pro_data/PreAF_RR_300.npy', threshold=0.24)

# 拼接特征矩阵与构建标签 (NSR=0, PreAF=1)
X = np.vstack((X_nsr, X_preaf))
y = np.hstack((np.zeros(len(X_nsr)), np.ones(len(X_preaf))))

# 按 8:2 的比例随机划分测试集和训练集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 模型训练 (使用XGBoost拟合融合特征)
clf = xgb.XGBClassifier(eval_metric='logloss')
clf.fit(X_train, y_train)

# 预测与结果评估
y_pred = clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"\n================ 评估结果 ================")
print(f"XGBoost 模型准确率: {accuracy:.4f}")
print("分类报告:")
# 输出精确率、灵敏度、F1-Score等全指标，全面验证特征融合效果
print(classification_report(y_test, y_pred))

处理 PreAF_RR_300.npy: 100%|██████████| 75/75 [00:35<00:00,  2.10it/s]


================ 评估结果 ================
XGBoost 模型准确率: 0.7667
分类报告:
              precision    recall  f1-score   support

         0.0       0.80      0.75      0.77        16
         1.0       0.73      0.79      0.76        14

    accuracy                           0.77        30
   macro avg       0.77      0.77      0.77        30
weighted avg       0.77      0.77      0.77        30

